# SmolVLM Invoice Extraction Pipeline

This notebook fine-tunes and evaluates SmolVLM for invoice field extraction using existing project utilities.

In [1]:
# Core imports
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

from scripts.preprocess import preprocess_csv_files, InvoiceImagePreprocessor
from scripts.smolvlm_model import SmolVLMInvoiceModel, DEFAULT_FIELDS


/opt/anaconda3/envs/dsan6500/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Paths (edit for your machine/data layout)
PROJECT_ROOT = Path('../finalproject_data').resolve()
RAW_DATA_DIR = PROJECT_ROOT / 'batch_1'
subfolders = ["batch1_1", "batch1_2", "batch1_3"]
#RAW_IMAGES_DIR = RAW_DATA_DIR / 'images'
OUTPUT_DIR = PROJECT_ROOT / 'output_images' / 'smolvlm'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GT_CSV_PATH = PROJECT_ROOT / "cleaned_invoices.csv"
PROCESSED_IMAGES_CSV_PATH = PROJECT_ROOT / 'combined_results.csv'
PROCESSED_IMAGE_DIR = OUTPUT_DIR / 'smolvlm_processed_images'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)

PROJECT_ROOT: /Users/minhhungtran/Documents/DSAN/DSAN6500/finalproject_data
OUTPUT_DIR: /Users/minhhungtran/Documents/DSAN/DSAN6500/finalproject_data/output_images/smolvlm


## Optional preprocessing
Run this block if you still need to generate cleaned ground truth and processed invoice images.
If you already have these artifacts, skip this cell.

In [3]:
# RUN_PREPROCESS = False

# if RUN_PREPROCESS:
#     # Example: adjust CSV filenames to your setup
#     gt_df = preprocess_csv_files(
#         input_path=RAW_DATA_DIR,
#         output_path=GT_CSV_PATH,
#         csv_files=['batch1_1.csv', 'batch1_2.csv', 'batch1_3.csv'],
#     )

#     preprocessor = InvoiceImagePreprocessor(output_dir=PROCESSED_IMAGE_DIR)
#     processed_images_df = preprocessor.process_images(
#         csv_path=str(GT_CSV_PATH),
#         image_folder_path=str(RAW_IMAGES_DIR),
#         batch_size=50,
#     )
#     processed_images_df.to_csv(PROCESSED_IMAGES_CSV_PATH, index=False)

#     print('Saved ground truth to', GT_CSV_PATH)
#     print('Saved processed images index to', PROCESSED_IMAGES_CSV_PATH)


In [4]:
# Load prepared data artifacts
ground_truth_df = pd.read_csv(GT_CSV_PATH)
processed_images_df = pd.read_csv(PROCESSED_IMAGES_CSV_PATH)

print('Ground truth rows:', len(ground_truth_df))
print('Processed image rows:', len(processed_images_df))
display(ground_truth_df.head(2))
display(processed_images_df.head(2))

Ground truth rows: 1413
Processed image rows: 1414


,File Name,OCRed Text,batch_csv,client_name,seller_name,invoice_number,invoice_date,due_date,tax,total_amount,net_worth
0,batch1-0494.jpg,Invoice no: 84652373 Date of issue: 02/23/2021...,batch1_1.csv,Clark-Foster,Nguyen-Roach,84652373,2021-02-23,NaN,21.18,232.95,211.77
1,batch1-0489.jpg,Invoice no: 37451664 Date of issue: 06/11/2020...,batch1_1.csv,"Williams, Schneider and Gomez",Scott-Howard,37451664,2020-06-11,NaN,13.99,153.92,139.93


,Unnamed: 0,original_file,processed_file,original_path,processed_path,status
0,0,batch1-0494.jpg,processed_batch1-0494.jpg,../finalproject_data/batch_1/batch1_1/batch1-0...,../finalproject_data/processed_images/processe...,success
1,1,batch1-0489.jpg,processed_batch1-0489.jpg,../finalproject_data/batch_1/batch1_1/batch1-0...,../finalproject_data/processed_images/processe...,success


In [5]:
# Initialize SmolVLM wrapper
smol = SmolVLMInvoiceModel(
    model_name='HuggingFaceTB/SmolVLM-256M-Instruct',
    output_dir=OUTPUT_DIR,
)

samples = smol.build_samples(
    ground_truth_df=ground_truth_df,
    processed_images_df=processed_images_df,
    fields=DEFAULT_FIELDS,
)

print('Training-ready samples:', len(samples))

Loading weights: 100%|██████████| 471/471 [00:00<00:00, 9698.79it/s]


Training-ready samples: 1414


In [6]:
# Train/validation/test split
# 80% train, 10% val, 10% test
train_samples, temp_samples = train_test_split(samples, test_size=0.2, random_state=42)
val_samples, test_samples = train_test_split(temp_samples, test_size=0.5, random_state=42)

print('Train:', len(train_samples), '| Val:', len(val_samples), '| Test:', len(test_samples))

# Build dataframe views for test-only inference/evaluation
test_keys = {s.processed_file for s in test_samples}
test_processed_images_df = processed_images_df[
    processed_images_df['processed_file'].astype(str).isin(test_keys)
].copy()

# Ground truth may have either File Name or processed_file; module handles both,
# but keeping a test-sliced frame makes intent explicit.
if 'processed_file' in ground_truth_df.columns:
    test_ground_truth_df = ground_truth_df[
        ground_truth_df['processed_file'].astype(str).isin(test_keys)
    ].copy()
else:
    test_ground_truth_df = ground_truth_df.copy()

print('Test processed rows:', len(test_processed_images_df))
print('Test ground truth rows:', len(test_ground_truth_df))

Train: 1131 | Val: 141 | Test: 142
Test processed rows: 142
Test ground truth rows: 1413


In [7]:
# Fine-tune SmolVLM (with quick-run options)
FAST_MODE = True

train_subset = train_samples
val_subset = val_samples

if FAST_MODE:
    # Use smaller subsets for rapid iteration/debugging
    train_subset, _ = train_test_split(train_samples, train_size=min(0.2, max(32 / len(train_samples), 0.05)), random_state=42)
    val_subset, _ = train_test_split(val_samples, train_size=min(0.5, max(16 / len(val_samples), 0.1)), random_state=42)

print('Training on:', len(train_subset), 'samples | Validating on:', len(val_subset), 'samples')

history = smol.train(
    train_samples=train_subset,
    val_samples=val_subset,
    epochs=1,
    batch_size=1 if FAST_MODE else 2,
    learning_rate=2e-5,
    max_label_length=128 if FAST_MODE else 256,
)

print(history)
model_dir = smol.save()
print('Saved model to', model_dir)

Training on: 56 samples | Validating on: 16 samples


SmolVLM train epoch 1/1: 100%|██████████| 56/56 [16:52:42<00:00, 1085.04s/it, loss=0.0527]  


{'train_loss': [0.11401386406006557], 'val_loss': [0.02427965129027143]}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.24it/s]

Saved model to /Users/minhhungtran/Documents/DSAN/DSAN6500/finalproject_data/output_images/smolvlm/model


In [8]:
# Test-set inference
pred_df = smol.predict_dataset(processed_images_df=test_processed_images_df)
print('Test prediction rows:', len(pred_df))
display(pred_df.head())

SmolVLM inference: 100%|██████████| 142/142 [3:30:39<00:00, 89.01s/it]  

Test prediction rows: 142


,processed_file,invoice_number,invoice_date,seller_name,client_name,tax,net_worth,total_amount
0,processed_batch1-0384.jpg,20295549,2011-12-20,Williams-Morris,Ward-Jordan,657.91,6579.11,7237.02
1,processed_batch1-0056.jpg,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,processed_batch1-0123.jpg,47037480,2012-09-03,Fernandez-Evans,Ruiz-Cowan,783.8,7837.98,7837.98
3,processed_batch1-0026.jpg,51295021,2017-01-05,Sims-Olson,Martinez-English,841.51,8415.08,9256.59
4,processed_batch1-0229.jpg,87958388,2016-04-11,Robinson and Sons,Jackson-Cardenas,463.8,4637.97,5101.77


In [9]:
# Evaluation on test set only
metrics_df, overall = smol.evaluate_against_ground_truth(
    ground_truth_df=test_ground_truth_df,
    pred_df=pred_df,
    fields=DEFAULT_FIELDS,
)

display(metrics_df)
print('Test overall metrics:', overall)

Key overlap: 142


,field,ground_truth_count,predicted_count,correct,accuracy,precision,recall,f1
0,invoice_number,142,139,130,0.915493,0.935252,0.915493,0.925267
1,invoice_date,142,139,134,0.943662,0.964029,0.943662,0.953737
2,seller_name,142,139,138,0.971831,0.992806,0.971831,0.982206
3,client_name,142,139,131,0.922535,0.942446,0.922535,0.932384
4,tax,142,129,122,0.859155,0.945736,0.859155,0.900369
5,net_worth,142,139,117,0.823944,0.841727,0.823944,0.832740
6,total_amount,142,139,120,0.845070,0.863309,0.845070,0.854093


Test overall metrics: {'accuracy': np.float64(0.89738430583501), 'precision': np.float64(0.9262720664589823), 'recall': np.float64(0.89738430583501), 'f1': np.float64(0.911599386816556)}


In [10]:
# Single-image inference example (from test set)
example_row = test_processed_images_df[test_processed_images_df['status'] == 'success'].iloc[0]
example_pred = smol.predict_single(image_path=example_row['processed_path'])
print('Test file:', example_row['processed_file'])
print('Prediction:', example_pred)

Test file: processed_batch1-0384.jpg
Prediction: {'invoice_number': '20295549', 'invoice_date': '2011-12-20', 'seller_name': 'Williams-Morris', 'client_name': 'Ward-Jordan', 'tax': '657.91', 'net_worth': '6579.11', 'total_amount': '7237.02'}
